In [0]:
%pip install lightgbm shap


In [0]:
# Imports e carregamento da tabela gold
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import lightgbm as lgb
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, roc_curve
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
import warnings
warnings.filterwarnings("ignore")

df_gold = spark.table("gold_episodios")
print("Total de episodios:", df_gold.count())
print("Colunas:", df_gold.columns)

In [0]:
# Amostra dos registros do gold 
df_gold.limit(5).display()


**Dicionário da Gold**

**Identificadores do episódio**
- `account_id` - id do cliente
- `offer_id` - id da oferta
- `episodio` - número do envio daquela oferta para aquele cliente (1º, 2º, 3º envio)

**Temporalidade**
- `received_time` - momento do recebimento (decimal, ex: 7.0 = dia 7)
- `received_day` - dia inteiro do recebimento (usado para split temporal)
- `viewed_time` / `viewed_day` - momento da visualização, se ocorreu dentro da validade
- `completed_time` / `completed_day` - momento da conclusão, se ocorreu dentro da validade
- `valid_until` - prazo de validade do episódio (`received_time + duration`)

**Conclusão**
- `reward` - valor do desconto efetivamente pago (somado quando houve mais de uma conclusão no mesmo episódio)
- `n_conclusoes_no_episodio` - quantas vezes o cliente concluiu a condição dentro da janela

**Metadados da oferta**
- `offer_type` - tipo da oferta: `bogo`, `discount` ou `informational`
- `discount_value` - valor do desconto em R$
- `min_value` - gasto mínimo para ativar a oferta
- `duration` - duração da oferta em dias
- `n_channels` - número de canais de veiculação (web, email, mobile, social)

**Perfil do cliente**
- `age` - idade do cliente (nulo para cadastros incompletos)
- `gender` - gênero: M, F ou O
- `credit_card_limit` - limite do cartão registrado
- `registered_on` - data de criação da conta
- `incomplete_profile` - flag 1/0 para cadastros com age=118 substituído por nulo

## Feature Engineering

Todas as features comportamentais são calculadas com dados anteriores ao `received_time` de cada episodio, sem data leakage.

`days_since_registration` usa data base `2018-07-26` (inicio do periodo de campanhas no dataset).

O histórico de visualizações e conclusões só conta eventos que já tinham ocorrido no momento do envio da oferta atual.

In [0]:
# Historico de compras: transacoes ANTES do received_time de cada episodio
df_transacoes = spark.table("silver_events") \
    .filter(F.col("event") == "transaction") \
    .select(
        "account_id",
        F.col("time_since_test_start").alias("transaction_time"),
        "amount"
    )

df_feat_compra = df_gold \
    .select("account_id", "offer_id", "episodio", "received_time") \
    .join(df_transacoes, on="account_id", how="left") \
    .filter(F.col("transaction_time") < F.col("received_time")) \
    .groupBy("account_id", "offer_id", "episodio", "received_time") \
    .agg(
        F.count("amount").alias("freq_anterior"),
        F.round(F.avg("amount"), 2).alias("ticket_medio_anterior"),
        F.round(F.sum("amount"), 2).alias("gasto_total_anterior"),
        F.round(F.max("transaction_time"), 2).alias("ultima_transacao_time")
    ) \
    .withColumn(
        "recencia",
        F.round(F.col("received_time") - F.col("ultima_transacao_time"), 2)
    )

# Historico promocional: viewed/completed contam so se ocorreram ANTES do received_time atual
df_episodios_hist = df_gold.select(
    "account_id", "offer_id", "episodio",
    "received_time", "viewed_time", "completed_time", "offer_type"
)

df_hist_ofertas = df_gold \
    .select("account_id", "offer_id", "episodio", "received_time", "offer_type") \
    .alias("atual") \
    .join(
        df_episodios_hist.alias("hist"),
        on=[
            F.col("atual.account_id") == F.col("hist.account_id"),
            F.col("hist.received_time") < F.col("atual.received_time")
        ],
        how="left"
    ) \
    .groupBy(
        F.col("atual.account_id"),
        F.col("atual.offer_id"),
        F.col("atual.episodio"),
        F.col("atual.received_time"),
        F.col("atual.offer_type")
    ) \
    .agg(
        F.count("hist.episodio").alias("n_ofertas_anteriores"),
        F.count(
            F.when(
                F.col("hist.viewed_time").isNotNull() &
                (F.col("hist.viewed_time") < F.col("atual.received_time")),
                1
            )
        ).alias("n_views_anteriores"),
        F.count(
            F.when(
                F.col("hist.completed_time").isNotNull() &
                (F.col("hist.completed_time") < F.col("atual.received_time")),
                1
            )
        ).alias("n_completadas_anteriores"),
        F.count(
            F.when(
                (F.col("hist.offer_type") == "bogo") &
                F.col("hist.completed_time").isNotNull() &
                (F.col("hist.completed_time") < F.col("atual.received_time")),
                1
            )
        ).alias("n_completadas_bogo_anterior"),
        F.count(
            F.when(
                (F.col("hist.offer_type") == "discount") &
                F.col("hist.completed_time").isNotNull() &
                (F.col("hist.completed_time") < F.col("atual.received_time")),
                1
            )
        ).alias("n_completadas_discount_anterior")
    ) \
    .withColumn(
        "taxa_view_historica",
        F.when(
            F.col("n_ofertas_anteriores") > 0,
            F.round(F.col("n_views_anteriores") / F.col("n_ofertas_anteriores"), 3)
        ).otherwise(None)
    ) \
    .withColumn(
        "taxa_conclusao_historica",
        F.when(
            F.col("n_ofertas_anteriores") > 0,
            F.round(F.col("n_completadas_anteriores") / F.col("n_ofertas_anteriores"), 3)
        ).otherwise(None)
    )

# Join final com features de interacao e days_since_registration corrigido
df_features_v2 = df_gold \
    .join(
        df_feat_compra.drop("received_time"),
        on=["account_id", "offer_id", "episodio"],
        how="left"
    ) \
    .join(
        df_hist_ofertas.drop("received_time", "offer_type"),
        on=["account_id", "offer_id", "episodio"],
        how="left"
    ) \
    .withColumn(
        "ticket_ratio",
        F.when(
            (F.col("min_value") > 0) & (F.col("ticket_medio_anterior").isNotNull()),
            F.round(F.col("ticket_medio_anterior") / F.col("min_value"), 3)
        ).otherwise(None)
    ) \
    .withColumn(
        "discount_ratio",
        F.when(
            F.col("min_value") > 0,
            F.round(F.col("discount_value") / F.col("min_value"), 3)
        ).otherwise(None)
    ) \
    .withColumn(
        "ticket_minus_minvalue",
        F.when(
            F.col("ticket_medio_anterior").isNotNull(),
            F.round(F.col("ticket_medio_anterior") - F.col("min_value"), 2)
        ).otherwise(None)
    ) \
    .withColumn(
        "days_since_registration",
        F.when(
            F.col("registered_on").isNotNull(),
            F.datediff(
                F.date_add(F.lit("2018-07-26"), F.col("received_day").cast("int")),
                F.col("registered_on")
            )
        ).otherwise(None)
    )

print("Features v2 geradas:", df_features_v2.count(), "| esperado 76277")

In [0]:
# Tratamento de nulos, encoding, targets, censura temporal e split
FEATURES_V2 = [
    "age", "gender_encoded", "credit_card_limit", "incomplete_profile",
    "days_since_registration",
    "freq_anterior", "ticket_medio_anterior", "gasto_total_anterior", "recencia",
    "n_ofertas_anteriores", "n_views_anteriores", "n_completadas_anteriores",
    "n_completadas_bogo_anterior", "n_completadas_discount_anterior",
    "taxa_view_historica", "taxa_conclusao_historica",
    "offer_type_encoded", "discount_value", "min_value", "duration", "n_channels",
    "ticket_ratio", "discount_ratio", "ticket_minus_minvalue"
]

TARGET   = "successful_response"
MAX_TIME = 29.75  # episodios com valid_until > 29.75 tem janela incompleta

df_model_v2 = df_features_v2 \
    .withColumn("freq_anterior",                    F.coalesce(F.col("freq_anterior"),                    F.lit(0))) \
    .withColumn("ticket_medio_anterior",            F.coalesce(F.col("ticket_medio_anterior"),            F.lit(0.0))) \
    .withColumn("gasto_total_anterior",             F.coalesce(F.col("gasto_total_anterior"),             F.lit(0.0))) \
    .withColumn("recencia",                         F.coalesce(F.col("recencia"),                         F.lit(999.0))) \
    .withColumn("n_ofertas_anteriores",             F.coalesce(F.col("n_ofertas_anteriores"),             F.lit(0))) \
    .withColumn("n_views_anteriores",               F.coalesce(F.col("n_views_anteriores"),               F.lit(0))) \
    .withColumn("n_completadas_anteriores",         F.coalesce(F.col("n_completadas_anteriores"),         F.lit(0))) \
    .withColumn("n_completadas_bogo_anterior",      F.coalesce(F.col("n_completadas_bogo_anterior"),      F.lit(0))) \
    .withColumn("n_completadas_discount_anterior",  F.coalesce(F.col("n_completadas_discount_anterior"),  F.lit(0))) \
    .withColumn("taxa_view_historica",              F.coalesce(F.col("taxa_view_historica"),              F.lit(0.0))) \
    .withColumn("taxa_conclusao_historica",         F.coalesce(F.col("taxa_conclusao_historica"),         F.lit(0.0))) \
    .withColumn("ticket_ratio",                     F.coalesce(F.col("ticket_ratio"),                     F.lit(0.0))) \
    .withColumn("discount_ratio",                   F.coalesce(F.col("discount_ratio"),                   F.lit(0.0))) \
    .withColumn("ticket_minus_minvalue",            F.coalesce(F.col("ticket_minus_minvalue"),            F.lit(0.0))) \
    .withColumn("gender_encoded",
        F.when(F.col("gender") == "M", 0)
         .when(F.col("gender") == "F", 1)
         .when(F.col("gender") == "O", 2)
         .otherwise(-1)
    ) \
    .withColumn("offer_type_encoded",
        F.when(F.col("offer_type") == "bogo",     0)
         .when(F.col("offer_type") == "discount", 1)
         .otherwise(-1)
    ) \
    .withColumn("successful_response",
        F.when(
            F.col("viewed_time").isNotNull() &
            F.col("completed_time").isNotNull() &
            (F.col("viewed_time") <= F.col("completed_time")),
            1
        ).otherwise(0)
    ) \
    .withColumn("completed_within_validity",
        F.when(F.col("completed_time").isNotNull(), 1).otherwise(0)
    ) \
    .withColumn("censurado",
        F.when(F.col("valid_until") > MAX_TIME, 1).otherwise(0)
    )

df_model_clean_v2 = df_model_v2 \
    .filter(F.col("offer_type") != "informational") \
    .filter(F.col("censurado") == 0)

colunas_pd = (
    ["account_id", "offer_id", "received_day"] +
    FEATURES_V2 +
    ["successful_response", "completed_within_validity"]
)

train_pd_v2 = df_model_clean_v2.filter(F.col("received_day").isin([0, 7])).select(colunas_pd).toPandas()
val_pd_v2   = df_model_clean_v2.filter(F.col("received_day") == 14).select(colunas_pd).toPandas()
test_pd_v2  = df_model_clean_v2.filter(F.col("received_day") == 17).select(colunas_pd).toPandas()

print("Treino:   ", train_pd_v2.shape)
print("Validacao:", val_pd_v2.shape)
print("Teste:    ", test_pd_v2.shape)

print("\nDistribuicao do target no treino:")
print(train_pd_v2[TARGET].value_counts().to_dict())

X_train = train_pd_v2[FEATURES_V2]
y_train = train_pd_v2[TARGET]
X_val   = val_pd_v2[FEATURES_V2]
y_val   = val_pd_v2[TARGET]
X_test  = test_pd_v2[FEATURES_V2]
y_test  = test_pd_v2[TARGET]

In [0]:
# Confirma que todas as ofertas estao representadas nos tres splits
colunas_cob = ["offer_id", "offer_type", "received_day", "successful_response"]

cob_spark = df_model_clean_v2 \
    .select(colunas_cob) \
    .withColumn("split",
        F.when(F.col("received_day").isin([0, 7]), "treino")
         .when(F.col("received_day") == 14, "validacao")
         .when(F.col("received_day") == 17, "teste")
         .otherwise("outro")
    ) \
    .filter(F.col("split") != "outro") \
    .groupBy("split", "offer_id", "offer_type") \
    .agg(
        F.count("*").alias("n_episodios"),
        F.round(F.avg("successful_response"), 3).alias("taxa_resposta")
    ) \
    .orderBy("offer_id", "split")

print("=== Cobertura de ofertas por split ===")
cob_spark.show(40, truncate=False)

ofertas_treino = set(cob_spark.filter(F.col("split") == "treino")
                    .select("offer_id").distinct().toPandas()["offer_id"])
ofertas_val    = set(cob_spark.filter(F.col("split") == "validacao")
                    .select("offer_id").distinct().toPandas()["offer_id"])
ofertas_teste  = set(cob_spark.filter(F.col("split") == "teste")
                    .select("offer_id").distinct().toPandas()["offer_id"])

print(f"Ofertas no treino:    {len(ofertas_treino)}")
print(f"Ofertas na validacao: {len(ofertas_val)}")
print(f"Ofertas no teste:     {len(ofertas_teste)}")
print(f"Apenas na validacao:  {ofertas_val - ofertas_treino or 'nenhuma'}")
print(f"Apenas no teste:      {ofertas_teste - ofertas_treino or 'nenhuma'}")

**Validação da Cobertura de Ofertas por Amostra**

Todas as 8 ofertas promocionais aparecem nos três conjuntos com volumes equilibrados e taxas de resposta estaveis entre splits. Não há oferta exclusiva de validação ou teste que o modelo nunca tenha visto no treino.

O modelo nao usa `offer_id` diretamente como feature. As ofertas são representadas por seus atributos (`offer_type`, `discount_value`, `min_value`, `duration`, `n_channels`), permitindo generalização para configurações semelhantes.

In [0]:
# Confirma criterio de censura: valid_until <= 29.75 (received_time + duration)
df_censura = df_model_v2 \
    .filter(F.col("offer_type") != "informational") \
    .withColumn(
        "valido",
        F.when(F.col("valid_until") <= MAX_TIME, "nao_censurado").otherwise("censurado")
    ) \
    .groupBy("offer_id", "offer_type", "duration", "valido") \
    .agg(
        F.count("*").alias("n_episodios"),
        F.round(F.avg("successful_response"), 3).alias("taxa_resposta")
    ) \
    .orderBy("offer_id", "valido")

print("=== Episodios por oferta, duracao e status de censura ===")
df_censura.show(40, truncate=False)

total_validos    = df_model_v2.filter(F.col("offer_type") != "informational") \
                              .filter(F.col("valid_until") <= MAX_TIME).count()
total_censurados = df_model_v2.filter(F.col("offer_type") != "informational") \
                              .filter(F.col("valid_until") > MAX_TIME).count()

print(f"Nao censurados (valid_until <= 29.75): {total_validos:,}")
print(f"Censurados    (valid_until > 29.75):   {total_censurados:,}")
print(f"Proporcao excluida: {total_censurados / (total_validos + total_censurados):.1%}")

**Critério de Censura Temporal**

Episódios onde `valid_until > 29.75` foram excluídos do modelo. A condição
equivale a `received_time + duration > 29.75` — ou seja, a janela de validade
da oferta ultrapassou o fim do experimento. Incluir esses casos classificaria
conversões ainda pendentes como negativos, introduzindo falsos negativos.

Ofertas de 10 dias perderam cerca de 33% dos episódios por censura, as de 7
dias cerca de 17%, e as de 5 dias nenhum.

Total excluído: 10.236 episódios (16,8%). Dataset final para modelagem: 50.806.

In [0]:
# Regressao Logistica como baseline de performance
# Exige escala padronizada e imputacao de nulos (LightGBM não precisa)
X_train_lr = X_train.copy()
X_val_lr   = X_val.copy()
X_test_lr  = X_test.copy()

for col in ["age", "credit_card_limit"]:
    med = X_train_lr[col].median()
    X_train_lr[col] = X_train_lr[col].fillna(med)
    X_val_lr[col]   = X_val_lr[col].fillna(med)
    X_test_lr[col]  = X_test_lr[col].fillna(med)

pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train_lr, y_train)

for nome, X, y in [("Validacao", X_val_lr, y_val), ("Teste", X_test_lr, y_test)]:
    probs = pipe_lr.predict_proba(X)[:, 1]
    preds = pipe_lr.predict(X)
    print(f"\n=== Regressao Logistica — {nome} ===")
    print(f"ROC-AUC: {roc_auc_score(y, probs):.4f}")
    print(f"PR-AUC:  {average_precision_score(y, probs):.4f}") # caso a variável resposta fosse desbalanceada (não é). só pra comparação; 
    print(f"Brier:   {brier_score_loss(y, probs):.4f}")
    print(classification_report(y, preds, digits=3))

In [0]:
# Modelo principal: Light GBM
# days_since_registration base: 26-07-2018

def avaliar(model, X, y, nome):
    probs      = model.predict_proba(X)[:, 1]
    n          = len(y)
    idx_ord    = np.argsort(probs)[::-1]
    y_ord      = np.array(y)[idx_ord]
    taxa_global = np.mean(y)
    return {
        "Conjunto":  nome,
        "ROC-AUC":   round(roc_auc_score(y, probs), 4),
        "PR-AUC":    round(average_precision_score(y, probs), 4),
        "Brier":     round(brier_score_loss(y, probs), 4),
        "Lift@10%":  round(y_ord[:int(0.1 * n)].mean() / taxa_global, 2),
        "Lift@20%":  round(y_ord[:int(0.2 * n)].mean() / taxa_global, 2),
    }

model_v3 = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    verbose=-1
)

model_v3.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
)

print("Melhor iteracao:", model_v3.best_iteration_)

resultados = [
    avaliar(model_v3, X_val,  y_val,  "LightGBM v3 — Validacao"),
    avaliar(model_v3, X_test, y_test, "LightGBM v3 — Teste"),
]
print(pd.DataFrame(resultados).to_string(index=False))

In [0]:
# Platt Scaling no conjunto de validacao; avaliado no teste

##calibrator = CalibratedClassifierCV(model_v3, method="sigmoid", cv="prefit")
##calibrator.fit(X_val, y_val)
calibrator = CalibratedClassifierCV(model_v3, method="sigmoid", cv=None, ensemble=False)
calibrator.fit(X_val, y_val)

probs_raw = model_v3.predict_proba(X_test)[:, 1]
probs_cal = calibrator.predict_proba(X_test)[:, 1]

print("=== Calibracao — Teste ===")
print(f"Brier sem calibracao: {brier_score_loss(y_test, probs_raw):.4f}")
print(f"Brier calibrado:      {brier_score_loss(y_test, probs_cal):.4f}")
print(f"PR-AUC sem:           {average_precision_score(y_test, probs_raw):.4f}")
print(f"PR-AUC calibrado:     {average_precision_score(y_test, probs_cal):.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
for probs, label, cor in [
    (probs_raw, "LightGBM sem calibracao", "#E63946"),
    (probs_cal, "LightGBM calibrado (Platt)", "#2A6CF0"),
]:
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=10)
    ax.plot(mean_pred, frac_pos, marker="o", label=label, color=cor)

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Calibracao perfeita")
ax.set_title("Curva de Calibracao — Teste", fontweight="bold")
ax.set_xlabel("Probabilidade prevista")
ax.set_ylabel("Frequencia real de positivos")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Não melhorou o Modelo. 

**Calibração das Probabilidades**

A curva de calibraçã mostrou que o LightGBM já produz probabilidades bem ajustadas para o banco de dados. A aplicação de Platt Scaling pioraria levemente o Brier Score (0.1750 para 0.1787) sem ganho em PR-AUC.

Decisão: o modelo sem calibracao adicional será usado no score financeiro. Em produção, reavaliaria a calibracao com dados de periodo mais longo.

In [0]:
# SHAP Values no conjunto de teste
explainer   = shap.TreeExplainer(model_v3)
shap_values = explainer.shap_values(X_test)

# LightGBM retorna lista [neg, pos] para classificacao binaria
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(10, 8))
shap.summary_plot(sv, X_test, feature_names=FEATURES_V2, plot_type="bar", show=False)
plt.title("Importancia das Features - SHAP (LightGBM)", fontweight="bold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_test, feature_names=FEATURES_V2, show=False)
plt.title("Distribuicao dos SHAP Values — Teste", fontweight="bold")
plt.tight_layout()
plt.show()

**Interpretabilidade dos SHAP Values**

`days_since_registration` é o sinal mais forte: clientes com mais tempo de contaapresentaram maior probabilidade prevista de resposta. Antiguidade na plataforma captura **engajamento de longo prazo de forma mais direta do que qualquer variável de comportamento recente**.

`n_channels` aparece como segunda feature mais importante, mas com uma ressalva importante: cada oferta tem um conjunto fixo de canais de comunicação. **Os canais são atributos da oferta, não variáveis independentes**. Não é possível separar o efeito do canal do efeito da oferta com esses dados (`n_channels`) funciona como proxy do alcance da campanha, não como medida isolada do impacto de cada canal.

Para separar efeitos por canal de forma confiável seria necessário um experimento onde a mesma oferta fosse enviada por configurações de canal distintas para grupos equivalentes de clientes.

`credit_card_limit`, `ticket_medio_anterior` e `age` concentram as posições 3 a 5. Clientes com maior poder aquisitivo e ticket "histórico" mais alto estiveram associados a maior propensão de resposta.

`discount_value` apresentou associação negativa com a probabilidade prevista. Logo, ofertas com desconto maior tiveram menor propensão estimada. Isso é consistente com o que a análise descritiva já mostrava: acessibilidade da oferta aparenta pesar mais do que o tamanho do incentivo.

As features de histórico promocional (`n_completadas_bogo_anterior`, `n_completadas_discount_anterior`, `discount_ratio`) contribuíram pouco, o que é esperado dado que a maioria dos episódios é o primeiro ou segundo envio para aquele cliente.



In [0]:
# Sistema de Recomendação
# Calcularemos o "Valor esperado da transação líquido do desconto": 
## Score = prob_resposta * (max(ticket_medio_anterior, min_value) - discount_value)

# Sendo: 
## prob_resposta: A probabilidade do modelo de que aquele cliente vai responder à oferta. Se é 0.8, significa 80% de chance de converter.
## ticket_medio_anterior: quanto o cliente gastou em média antes do envio
## min_value: o gasto mínimo para ativar a oferta
## discount_value: O custo do desconto que o iFood vai pagar se o cliente converter.

# Sem normalizacao por cliente: a probabilidade absoluta pesa diretamente

# Catalogo de ofertas promocionais (extraido do conjunto de teste)
ofertas_pd = (
    df_model_clean_v2
    .filter(F.col("received_day") == 17)
    .select("offer_id", "offer_type", "offer_type_encoded",
            "discount_value", "min_value", "duration", "n_channels")
    .distinct()
    .toPandas()
)

# Perfil de cada cliente: uma linha por account_id (features de comportamento)
FEATURES_CLIENTE = [
    "account_id", "age", "gender_encoded", "credit_card_limit", "incomplete_profile",
    "days_since_registration", "freq_anterior", "ticket_medio_anterior",
    "gasto_total_anterior", "recencia", "n_ofertas_anteriores", "n_views_anteriores",
    "n_completadas_anteriores", "n_completadas_bogo_anterior",
    "n_completadas_discount_anterior", "taxa_view_historica", "taxa_conclusao_historica"
]
clientes_teste = (
    test_pd_v2[FEATURES_CLIENTE]
    .drop_duplicates("account_id")
    .reset_index(drop=True)
)

# Cross-join: cada cliente x cada oferta
linhas = []
for _, oferta in ofertas_pd.iterrows():
    temp = clientes_teste.copy()
    temp["offer_id"]           = oferta["offer_id"]
    temp["offer_type"]         = oferta["offer_type"]
    temp["offer_type_encoded"] = oferta["offer_type_encoded"]
    temp["discount_value"]     = oferta["discount_value"]
    temp["min_value"]          = oferta["min_value"]
    temp["duration"]           = oferta["duration"]
    temp["n_channels"]         = oferta["n_channels"]
    # Recalcula features de interacao para a oferta corrente
    mv = oferta["min_value"]
    temp["ticket_ratio"]          = np.where(mv > 0, (temp["ticket_medio_anterior"] / mv).clip(lower=0), 0)
    temp["discount_ratio"]        = oferta["discount_value"] / mv if mv > 0 else 0
    temp["ticket_minus_minvalue"] = (temp["ticket_medio_anterior"] - mv).clip(lower=0)
    linhas.append(temp)

df_rec = pd.concat(linhas, ignore_index=True)
df_rec["prob_resposta"] = model_v3.predict_proba(df_rec[FEATURES_V2])[:, 1]

df_rec["valor_base"] = np.maximum(
    df_rec["ticket_medio_anterior"].clip(lower=0),
    df_rec["min_value"]
)
df_rec["score_financeiro"] = (
    df_rec["prob_resposta"] * (df_rec["valor_base"] - df_rec["discount_value"])
)

# Melhor oferta por cliente
idx_melhor = df_rec.groupby("account_id")["score_financeiro"].idxmax()
df_recomendacao = df_rec.loc[idx_melhor].copy()

# Nao recomenda se a maior probabilidade do cliente for < 0.30
prob_max = df_rec.groupby("account_id")["prob_resposta"].max()
df_recomendacao["nao_enviar"] = df_recomendacao["account_id"].map(prob_max < 0.30)
df_recomendacao["recomendacao_final"] = np.where(
    df_recomendacao["nao_enviar"], "nao_enviar", df_recomendacao["offer_id"]
)

print("Distribuicao das recomendacoes:")
print(df_recomendacao["recomendacao_final"].value_counts())
print(f"\nClientes sem recomendacao (prob max < 0.30): {df_recomendacao['nao_enviar'].sum()}")

print("\nProbabilidade da melhor oferta por cliente:")
print(df_recomendacao["prob_resposta"].describe().round(3))

# Probabilidade media por oferta no conjunto de teste
print("\nProb media de resposta por oferta (todos os clientes):")
print(df_rec.groupby("offer_id")["prob_resposta"].mean().sort_values(ascending=False).round(3))

**Sistema de Recomendação** 

Para cada cliente, o modelo gera uma probabilidade de resposta para cada uma das 8 ofertas promocionais. A oferta recomendada é escolhida pelo **maior valor líquido promocional** estimado:

_score = P(resposta) × (max(ticket_medio_cliente, min_value) − discount_value)_

O `max(ticket_medio, min_value)` estima o gasto esperado caso o cliente converta. Se o ticket habitual já supera o valor mínimo, ele tende a gastar o que costuma. Caso contrário, ele vai precisar gastar pelo menos o mínimo para ativar a oferta. O desconto é subtraído como custo do incentivo.

- Clientes com probabilidade máxima abaixo de 0.30 não recebem recomendação.

**Resultados no conjunto de teste (onda dia 17, n = 10.210):**

- 679 clientes (6,6%) classificados como "não enviar"
- 9.460 clientes (92,7%) receberam a oferta `fafdcd668` (discount R$2, 10 dias,  4 canais) como recomendação. Ela foi a de maior probabilidade média observada (0.647)
- As demais ofertas foram recomendadas para 71 clientes no total

A concentração em uma única oferta não é um problema do modelo. É uma limitação do delineamento. Como cada cliente recebeu** apenas uma oferta por onda**, nunca observamos o que aconteceria se aquele cliente tivesse recebido uma oferta diferente. O modelo aprende o efeito médio de cada oferta, não a compatibilidade individual cliente-oferta. Para personalizar a escolha da oferta de forma confiável, o experimento precisaria randomizar qual oferta cada cliente recebe dentro de cada onda.

In [0]:
# Agrupando episodios do teste por faixa de propensão
test_pd_v2_rec = test_pd_v2.copy()
test_pd_v2_rec["prob_v2"]       = model_v3.predict_proba(X_test)[:, 1]
test_pd_v2_rec["custo_esperado"] = test_pd_v2_rec["prob_v2"] * test_pd_v2_rec["discount_value"]
test_pd_v2_rec["valor_base"]     = np.maximum(
    test_pd_v2_rec["ticket_medio_anterior"].clip(lower=0),
    test_pd_v2_rec["min_value"]
)
test_pd_v2_rec["valor_liquido_est"] = (
    test_pd_v2_rec["prob_v2"] *
    (test_pd_v2_rec["valor_base"] - test_pd_v2_rec["discount_value"])
)

test_pd_v2_rec["faixa_propensao"] = pd.cut(
    test_pd_v2_rec["prob_v2"],
    bins=[0, 0.30, 0.55, 0.75, 1.0],
    labels=["Baixa (< 30%)", "Media (30-55%)", "Alta (55-75%)", "Muito alta (> 75%)"]
)

resumo = test_pd_v2_rec.groupby("faixa_propensao", observed=True).agg(
    n_clientes       = ("prob_v2",              "count"),
    prob_media        = ("prob_v2",              "mean"),
    ticket_medio      = ("ticket_medio_anterior", "mean"),
    freq_media        = ("freq_anterior",         "mean"),
    recencia_media    = ("recencia",              lambda x: x[x < 999].mean()),
    taxa_conclusao    = ("taxa_conclusao_historica", "mean"),
    dias_cadastro     = ("days_since_registration", "mean"),
    credito_medio     = ("credit_card_limit",     "mean"),
    valor_liq_medio   = ("valor_liquido_est",     "mean"),
    conversoes_obs    = ("successful_response",   "sum")
).reset_index()

resumo["pct_base"]          = (resumo["n_clientes"] / resumo["n_clientes"].sum() * 100).round(1)
resumo["taxa_conversao_obs"] = (resumo["conversoes_obs"] / resumo["n_clientes"] * 100).round(1)

print("=== Resumo por Faixa de Propensao ===")
cols = ["faixa_propensao", "n_clientes", "pct_base", "prob_media", "taxa_conversao_obs",
        "ticket_medio", "freq_media", "recencia_media", "taxa_conclusao",
        "dias_cadastro", "credito_medio", "valor_liq_medio"]
print(resumo[cols].round(2).to_string(index=False))

In [0]:
# Visualizacao do perfil de cada faixa de propensao
fig = plt.figure(figsize=(20, 16))
fig.suptitle("Perfil dos Publicos por Faixa de Propensao",
             fontsize=16, fontweight="bold", y=0.98)
gs   = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.4)
faixas = ["Baixa (< 30%)", "Media (30-55%)", "Alta (55-75%)", "Muito alta (> 75%)"]
cores  = ["#E63946", "#F4A261", "#2A6CF0", "#1D3557"]
labels_curtos = ["Baixa", "Media", "Alta", "M.Alta"]

def bar_faixa(ax, vals, titulo, ylabel, fmt="{:.1f}"):
    ax.bar(labels_curtos, vals, color=cores, edgecolor="white", alpha=0.9)
    for i, v in enumerate(vals):
        ax.text(i, v * 1.03, fmt.format(v), ha="center", fontsize=8)
    ax.set_title(titulo, fontweight="bold")
    ax.set_ylabel(ylabel)

# 1. Distribuicao de probabilidade
ax1 = fig.add_subplot(gs[0, 0])
for faixa, cor in zip(faixas, cores):
    sub = test_pd_v2_rec[test_pd_v2_rec["faixa_propensao"] == faixa]["prob_v2"]
    ax1.hist(sub, bins=20, alpha=0.6, color=cor, label=faixa.split(" ")[0], density=True)
ax1.set_title("Distribuicao de Probabilidade\npor Faixa", fontweight="bold")
ax1.set_xlabel("P(resposta)")
ax1.set_ylabel("Densidade")
ax1.legend(fontsize=7)

# 2. Ticket medio
bar_faixa(fig.add_subplot(gs[0, 1]),
          resumo["ticket_medio"].values, "Ticket Medio Anterior\npor Faixa", "R$", "R${:.1f}")

# 3. Frequencia de compras
bar_faixa(fig.add_subplot(gs[0, 2]),
          resumo["freq_media"].values, "Frequencia Media de Compras\npor Faixa", "Transacoes", "{:.1f}")

# 4. Boxplot ticket_ratio
ax4 = fig.add_subplot(gs[1, 0])
dados_box = [
    test_pd_v2_rec[test_pd_v2_rec["faixa_propensao"] == f]["ticket_ratio"].clip(0, 10)
    for f in faixas
]

#bp = ax4.boxplot(dados_box, patch_artist=True, tick_labels=labels_curtos)
bp = ax4.boxplot(dados_box, patch_artist=True, labels=labels_curtos)
for patch, cor in zip(bp["boxes"], cores):
    patch.set_facecolor(cor); patch.set_alpha(0.7)
ax4.set_title("Ticket Ratio por Faixa\n(ticket / min_value, cap=10)", fontweight="bold")
ax4.set_ylabel("Ticket Ratio")
ax4.axhline(1, color="gray", linestyle="--", linewidth=1, alpha=0.7)

# 5. Dias de cadastro
bar_faixa(fig.add_subplot(gs[1, 1]),
          resumo["dias_cadastro"].values, "Tempo Medio de Cadastro\npor Faixa (dias)", "Dias", "{:.0f}d")

# 6. Taxa historica de conclusao
bar_faixa(fig.add_subplot(gs[1, 2]),
          (resumo["taxa_conclusao"] * 100).values,
          "Taxa Historica de Conclusao\npor Faixa", "Taxa (%)", "{:.1f}%")

# 7. Valor liquido estimado
bar_faixa(fig.add_subplot(gs[2, 0]),
          resumo["valor_liq_medio"].values,
          "Valor Liquido Estimado Medio\npor Faixa", "R$", "R${:.1f}")

# 8. Limite de credito
bar_faixa(fig.add_subplot(gs[2, 1]),
          (resumo["credito_medio"] / 1000).values,
          "Limite de Credito Medio\npor Faixa", "R$ (mil)", "R${:.0f}k")

# 9. Taxa de conversao observada
bar_faixa(fig.add_subplot(gs[2, 2]),
          resumo["taxa_conversao_obs"].values,
          "Taxa de Conversao Observada\npor Faixa", "Taxa (%)", "{:.1f}%")

plt.tight_layout()
plt.show()

In [0]:
# Arvore de decisao rasa: traducao das regras de segmentacao em linguagem de negocio
# Nao substitui o LightGBM; serve apenas para explicar o que diferencia os grupos
FEATURES_ARVORE = [
    "days_since_registration", "ticket_medio_anterior", "freq_anterior",
    "recencia", "taxa_conclusao_historica", "ticket_ratio", "credit_card_limit"
]

le      = LabelEncoder()
y_faixa = le.fit_transform(test_pd_v2_rec["faixa_propensao"].astype(str))
X_arvore = test_pd_v2_rec[FEATURES_ARVORE].fillna(0)

tree = DecisionTreeClassifier(max_depth=3, random_state=42, min_samples_leaf=100)
tree.fit(X_arvore, y_faixa)

print("=== Arvore de Decisao Rasa — Regras Explicativas ===")
print(export_text(tree, feature_names=FEATURES_ARVORE))
print(f"Acuracia da arvore: {tree.score(X_arvore, y_faixa):.2f}")

# Tabela de publicos para negocio
tabela_negocio = pd.DataFrame({
    "Campanha":           ["Muito Alta (> 75%)", "Alta (55-75%)", "Media (30-55%)", "Baixa (< 30%)"],
    "N_clientes":         [1174, 2196, 2955, 3885],
    "Pct_base":           ["11.5%", "21.5%", "28.9%", "38.1%"],
    "Taxa_conv_obs":      ["77.5%", "62.5%", "40.9%", "11.7%"],
    "Ticket_medio":       ["R$17.3", "R$19.3", "R$14.3", "R$4.7"],
    "Dias_cadastro":      ["617d", "379d", "272d", "246d"],
    "Acao_recomendada":   [
        "Prioridade maxima — enviar sempre",
        "Alta prioridade — threshold 0.55",
        "Media — avaliar custo-beneficio",
        "Nao enviar — baixo retorno esperado"
    ]
})
print("\n=== Tabela de Publicos por Campanha ===")
print(tabela_negocio.to_string(index=False))

In [0]:
# Visualizacao grafica da arvore de decisao
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    tree,
    feature_names=FEATURES_ARVORE,
    class_names=le.classes_,
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax,
    impurity=False,
    proportion=False
)
ax.set_title(
    "Regras Explicativas — Arvore de Decisao Rasa (profundidade 3, apenas para interpretacao)",
    fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.show()

In [0]:
# Business case com reward real observado e custo esperado = prob * discount_value
df_teste_spark = df_model_clean_v2.filter(F.col("received_day") == 17)

reward_historico  = df_teste_spark \
    .filter(F.col("completed_within_validity") == 1) \
    .agg(F.sum("reward").alias("r")).collect()[0]["r"]

total_episodios   = df_teste_spark.count()
total_conversoes  = df_teste_spark.filter(F.col("successful_response") == 1).count()
sem_view          = df_teste_spark \
    .filter(F.col("completed_within_validity") == 1) \
    .filter(F.col("viewed_time").isNull()).count()

print("=== Politica Historica — Onda Teste (dia 17) ===")
print(f"Total de episodios:         {total_episodios:,}")
print(f"Conversoes:                 {total_conversoes:,}  ({total_conversoes/total_episodios:.1%})")
print(f"Reward total pago:          R$ {reward_historico:,.0f}")
print(f"Conclusoes sem view:        {sem_view:,}  ({sem_view/total_episodios:.1%})")

print("\n=== Simulacao por Threshold (custo esperado = prob * discount_value) ===")
rows = []
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    sel      = test_pd_v2_rec[test_pd_v2_rec["prob_v2"] >= t]
    n_sel    = len(sel)
    conv_obs = sel["successful_response"].sum()
    custo_esp = sel["custo_esperado"].sum()
    pct_conv = conv_obs / total_conversoes if total_conversoes > 0 else 0
    rows.append({
        "Threshold":      t,
        "Selecionados":   n_sel,
        "Reducao_%":      f"{(total_episodios - n_sel)/total_episodios:.1%}",
        "Conv_obs":       int(conv_obs),
        "Conv_pres_%":    f"{pct_conv:.1%}",
        "Custo_esp":      f"R$ {custo_esp:,.0f}",
        "Economia_pot":   f"R$ {reward_historico - custo_esp:,.0f}",
    })

print(pd.DataFrame(rows).to_string(index=False))
print("\nSimulacao retrospectiva. Ganhos reais devem ser validados via teste A/B em producao.")

In [0]:
# Lift@K e curvas de avaliacao do modelo no conjunto de teste
probs_teste = model_v3.predict_proba(X_test)[:, 1]
y_teste     = y_test.values

idx_ord   = np.argsort(probs_teste)[::-1]
y_ord     = y_teste[idx_ord]
n         = len(y_teste)
taxa_pos  = y_teste.mean()

conversoes_acum  = np.cumsum(y_ord)
percentis        = np.arange(0.01, 1.01, 0.01)
lift_values      = [(conversoes_acum[int(p * n) - 1] / int(p * n)) / taxa_pos for p in percentis]

fpr, tpr, _ = roc_curve(y_teste, probs_teste)
ks_stat     = np.max(tpr - fpr)

print(f"KS Statistic: {ks_stat:.4f}")
print(f"Lift@10%:     {lift_values[9]:.2f}x")
print(f"Lift@20%:     {lift_values[19]:.2f}x")
print(f"Lift@30%:     {lift_values[29]:.2f}x")
print(f"Lift@50%:     {lift_values[49]:.2f}x")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Avaliacao do Modelo — LightGBM v3", fontsize=14, fontweight="bold")

# Curva de Lift
ax1 = axes[0]
ax1.plot(percentis, lift_values, color="#2A6CF0", linewidth=2, label="LightGBM v3")
ax1.axhline(1, color="gray", linestyle="--", linewidth=1, label="Baseline aleatorio")
ax1.fill_between(percentis, 1, lift_values, alpha=0.15, color="#2A6CF0")
ax1.set_title("Curva de Lift", fontweight="bold")
ax1.set_xlabel("Percentil da populacao contatada")
ax1.set_ylabel("Lift (vezes melhor que aleatorio)")
ax1.legend()
ax1.grid(alpha=0.3)

# Curva de Ganho Acumulado
ax2 = axes[1]
ganho_modelo   = conversoes_acum / conversoes_acum[-1]
ganho_aleatorio = np.linspace(0, 1, n)
ax2.plot(np.linspace(0, 1, n), ganho_modelo, color="#2A6CF0", linewidth=2, label="LightGBM v3")
ax2.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1, label="Baseline aleatorio")
ax2.plot([0, taxa_pos, 1], [0, 1, 1], color="#E63946", linestyle="--", linewidth=1, label="Modelo perfeito")
ax2.set_title("Curva de Ganho Acumulado", fontweight="bold")
ax2.set_xlabel("Fracao da populacao contatada")
ax2.set_ylabel("Fracao das conversoes capturadas")
ax2.legend()
ax2.grid(alpha=0.3)

# Distribuicao de probabilidades por classe
ax3 = axes[2]
ax3.hist(probs_teste[y_teste == 0], bins=40, alpha=0.6, color="#E63946", label="Nao converteu", density=True)
ax3.hist(probs_teste[y_teste == 1], bins=40, alpha=0.6, color="#2A6CF0", label="Converteu",    density=True)
ax3.axvline(0.5, color="gray", linestyle="--", linewidth=1)
ax3.set_title("Distribuicao de Probabilidades\npor Classe Real", fontweight="bold")
ax3.set_xlabel("P(resposta)")
ax3.set_ylabel("Densidade")
ax3.legend()
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# Tabela consolidada de metricas: Baseline vs LightGBM v3
from sklearn.metrics import classification_report

print("Metricas Consolidadas (do Conjunto Teste)\n")

for nome, model, X, imputa in [
    ("Regressao Logistica", pipe_lr, X_test_lr, False),
    ("LightGBM v3",         model_v3, X_test,    False)
]:
    probs = model.predict_proba(X)[:, 1]
    preds = model.predict(X)
    print(f"--- {nome} ---")
    print(f"ROC-AUC:     {roc_auc_score(y_test, probs):.4f}")
    print(f"PR-AUC:      {average_precision_score(y_test, probs):.4f}")
    print(f"Brier Score: {brier_score_loss(y_test, probs):.4f}")
    print(classification_report(y_test, preds, digits=3))


## Conclusão

O objetivo do case era desenvolver uma técnica que auxiliasse na decisão de **qual oferta priorizar para cada cliente** e demonstrar seu potencial impacto no negócio.

A solução construída atua em dois níveis:

1. estima a probabilidade de resposta de cada combinação cliente-oferta;
2. utiliza essa probabilidade, junto ao custo promocional e ao valor esperado, para apoiar a priorização dos envios.

### Métricas no conjunto de teste

Onda do dia 17, com 10.210 episódios.

| Métrica               | Regressão Logística | LightGBM v3 |
| --------------------- | ------------------: | ----------: |
| ROC-AUC               |               0.760 |   **0.804** |
| PR-AUC                |               0.639 |   **0.691** |
| Brier Score           |               0.194 |   **0.175** |
| F1 da classe positiva |               0.533 |   **0.658** |
| KS Statistic          |                   — |   **0.463** |
| Lift@10%              |                   — |   **2.03x** |

Os resultados indicam boa capacidade de priorização. Entre os 10% de clientes com maior propensão prevista, a taxa de resposta foi aproximadamente duas vezes superior à seleção aleatória. Nos 40% mais bem ranqueados, o modelo concentrou cerca de 70% das respostas observadas no período.

### Aplicação prática

Na simulação retrospectiva com threshold de 0,30, a política proposta apresentou:

* redução aproximada de 39% no volume de envios;
* preservação de 87,8% das respostas observadas;
* custo promocional esperado de R$ 17.515, comparado ao reward histórico observado de R$ 29.398 na onda analisada.

Além disso, 679 clientes, equivalentes a 6,6% da base de teste, apresentaram probabilidade máxima inferior a 30% entre todas as ofertas avaliadas e seriam despriorizados pela política simulada.

Esses resultados representam uma estimativa offline e devem ser validados por meio de teste A/B antes de qualquer implantação.

### Limitação principal

O histórico disponível não permite identificar causalmente qual oferta seria melhor para cada cliente.

Em cada onda, observamos apenas o resultado da oferta efetivamente recebida, sem observar o resultado contrafactual das demais alternativas. Como consequência, o modelo capturou fortemente o desempenho médio das características das ofertas, o que contribuiu para a concentração das recomendações em uma única opção.

Portanto, a principal contribuição da solução está na **priorização de clientes com maior potencial de resposta e valor**, enquanto a escolha individual da oferta deve ser interpretada como apoio à decisão, e não como personalização causal comprovada.

### Possíveis evoluções

Com um desenho experimental que distribua ofertas de forma randomizada entre perfis comparáveis e mantenha um grupo de controle, seria possível avançar para abordagens mais robustas de personalização.

* **Uplift Modeling ou Causal ML:** estimar o efeito incremental de cada oferta em relação ao cenário sem incentivo.

* **Learning to Rank:** ordenar as ofertas por cliente quando houver histórico suficiente de comparações entre alternativas.

* **Contextual Bandit:** aprender continuamente qual oferta maximiza valor para cada contexto, equilibrando exploração e aproveitamento.

* **Modelos de recomendação latente:** podem ser considerados caso exista maior volume de interações repetidas entre clientes e ofertas, condição necessária para justificar técnicas como Matrix Factorization.

Em produção, a próxima etapa recomendada é validar a política por meio de experimento controlado, monitorando conversão incremental, valor líquido promocional e custo de desconto.
